# 과제: 시계열 예측 모델을 활용한 트레이딩 전략 개발 (All-in-One 버전)

**목표:** Buy and Hold 벤치마크를 초과하는 수익률을 내는 딥러닝 모델 기반 트레이딩 전략을 개발합니다.

**참고:** 이 노트북은 외부 `utils.py` 파일 없이, 모든 코드가 포함된 독립적인 파일입니다. Colab에 이 파일 하나만 업로드하여 실행할 수 있습니다.

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# Colab 환경에서 필요한 패키지를 설치합니다.
!pip install yfinance ta -q

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from ta.momentum import RSIIndicator
from ta.trend import MACD
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 기본 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 5)

print(f"Using device: {device}")

## 2. 필요 함수 정의 (과거 `utils.py`의 내용)

데이터 로딩, 특성 생성 등 필요한 함수들을 노트북 내에 직접 정의합니다.

In [ ]:
def load_bitcoin_data(start_date='2020-01-01', end_date=None):
    """yfinance를 사용하여 비트코인 데이터를 다운로드합니다."""
    if end_date is None:
        end_date = datetime.now().strftime('%Y-%m-%d')
    print(f"비트코인 데이터 다운로드 중: {start_date} ~ {end_date}")
    btc_data = yf.download('BTC-USD', start=start_date, end=end_date)
    print(f"다운로드 완료: {len(btc_data)} 행")
    return btc_data

def create_features(df):
    """가격 데이터로부터 특성을 생성합니다."""
    data = df.copy()
    close = data['Close']

    # 이동평균
    for window in [5, 10, 20, 50]:
        ma = close.rolling(window=window).mean()
        data[f'MA_{window}_ratio'] = close / ma
    
    # RSI
    data['RSI_14'] = RSIIndicator(close, window=14).rsi()
    
    # MACD
    macd_indicator = MACD(close)
    data['MACD_diff'] = macd_indicator.macd_diff()
    
    # 과거 수익률 (lag features)
    returns = close.pct_change()
    for lag in range(1, 8):
        data[f'Returns_Lag_{lag}'] = returns.shift(lag)
    
    # 타겟 변수: 다음 날 가격이 오를지 (1) 내릴지 (0)
    data['Target'] = (close.shift(-1) > close).astype(int)
    
    return data

def calculate_buy_and_hold_return(prices, initial_capital=10000, transaction_fee=0.001):
    """Buy and Hold 전략의 수익률 계산"""
    btc_amount = (initial_capital * (1 - transaction_fee)) / prices[0]
    final_value = btc_amount * prices[-1] * (1 - transaction_fee)
    total_return = (final_value - initial_capital) / initial_capital * 100
    return {
        'final_value': final_value,
        'total_return': total_return,
    }

## 3. 데이터 준비 및 전처리

In [ ]:
# 데이터 로드 및 특성 생성
btc_df = load_bitcoin_data(start_date='2020-01-01')
data_featured = create_features(btc_df)
data_final = data_featured.dropna()

# 특성(X)과 타겟(y) 분리
feature_columns = [col for col in data_final.columns if col not in ['Target', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
X_raw = data_final[feature_columns]
y_raw = data_final['Target']

# 데이터 정규화
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

# 시퀀스 데이터 생성
def create_sequences(X_data, y_data, seq_length):
    xs, ys = [], []
    for i in range(len(X_data) - seq_length):
        x = X_data[i:(i + seq_length)]
        y = y_data.iloc[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 14
X_seq, y_seq = create_sequences(X_scaled, y_raw, SEQ_LENGTH)

print(f"입력 데이터 형태 (X): {X_seq.shape}")
print(f"타겟 데이터 형태 (y): {y_seq.shape}")

## 4. 데이터 분할 및 DataLoader 생성

In [ ]:
# 데이터 분할 (Train: 70%, Validation: 15%, Test: 15%)
train_size = int(len(X_seq) * 0.7)
val_size = int(len(X_seq) * 0.15)

X_train, y_train = X_seq[:train_size], y_seq[:train_size]
X_val, y_val = X_seq[train_size:train_size+val_size], y_seq[train_size:train_size+val_size]
X_test, y_test = X_seq[train_size+val_size:], y_seq[train_size+val_size:]

# PyTorch Tensor로 변환
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).view(-1, 1)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.FloatTensor(y_val).view(-1, 1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).view(-1, 1)

# DataLoader 생성
BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

print(f"데이터 로더 준비 완료! Train: {len(X_train)}개, Validation: {len(X_val)}개, Test: {len(X_test)}개")

## 5. GRU 모델 정의 및 학습

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(GRUClassifier, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h0 = torch.zeros(self.gru.num_layers, x.size(0), self.gru.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

# 모델 하이퍼파라미터
INPUT_SIZE = X_train.shape[2]
HIDDEN_SIZE = 64
NUM_LAYERS = 2
OUTPUT_SIZE = 1
DROPOUT = 0.3

# 모델 인스턴스 생성
model = GRUClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE, DROPOUT).to(device)
print(model)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=100, lr=0.001, patience=15):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    print("모델 학습을 시작합니다...")
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()
        
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                predicted = (outputs > 0.5).float()
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total
        avg_val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
                
    if best_model_state:
        model.load_state_dict(best_model_state)
        
    print("\n✅ 모델 학습 완료!")
    return history

In [ ]:
# 모델 학습 실행
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    lr=0.001,
    patience=15
)

## 6. 트레이딩 시뮬레이션 및 결과 분석

In [ ]:
def predict_with_probability(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch_X, _ in loader:
            batch_X = batch_X.to(device)
            probs = model(batch_X)
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_probs)

def simulate_trading(predictions_prob, actual_prices, threshold=0.5, position_scaling=False, initial_capital=10000, transaction_fee=0.001):
    capital = initial_capital
    btc_held = 0
    portfolio_values = [initial_capital]
    
    for i in range(len(predictions_prob) - 1):
        prob = predictions_prob[i][0]
        current_price = actual_prices[i]
        
        # 포지션 종료 (매도)
        if btc_held > 0:
            capital = btc_held * current_price * (1 - transaction_fee)
            btc_held = 0

        # 포지션 진입 (매수)
        if prob > threshold:
            investment_ratio = (prob - threshold) / (1 - threshold) if position_scaling else 1.0
            amount_to_invest = capital * investment_ratio
            btc_to_buy = (amount_to_invest / current_price) * (1 - transaction_fee)
            btc_held += btc_to_buy
            capital -= amount_to_invest
            
        portfolio_values.append(capital + btc_held * actual_prices[i+1])

    final_value = portfolio_values[-1]
    total_return = (final_value / initial_capital - 1) * 100

    return {
        "portfolio_values": portfolio_values,
        "final_value": final_value,
        "total_return": total_return
    }

In [ ]:
# 시뮬레이션용 실제 가격 데이터 준비
test_start_index = train_size + val_size + SEQ_LENGTH
simulation_prices = data_final['Close'][test_start_index:].values
simulation_dates = data_final.index[test_start_index:]

# 모델 예측 확률값 추출
model_probs = predict_with_probability(model, test_loader)

# 트레이딩 시뮬레이션 실행
model_result = simulate_trading(
    predictions_prob=model_probs,
    actual_prices=simulation_prices,
    threshold=0.55, # 보수적 접근을 위해 임계값 조정
    position_scaling=True # 확률 기반 투자 비중 조절
)

# Buy and Hold 전략 계산
buy_hold_result = calculate_buy_and_hold_return(prices=simulation_prices)

print("---*--- FINAL RESULTS ---*---")
print(f"[Buy and Hold] Final Asset: ${buy_hold_result['final_value']:,.2f} | Return: {buy_hold_result['total_return']:.2f}%")
print(f"[GRU Model]    Final Asset: ${model_result['final_value']:,.2f} | Return: {model_result['total_return']:.2f}%")
print("---*---*---*---*---*---*---")

In [ ]:
# 최종 결과 시각화
plt.figure(figsize=(18, 8))

# Buy and Hold 포트폴리오 가치
buy_hold_portfolio = 10000 * (simulation_prices / simulation_prices[0])

sim_dates_adjusted = simulation_dates[:len(model_result['portfolio_values'])]

plt.plot(simulation_dates, buy_hold_portfolio, label=f"Buy & Hold ({buy_hold_result['total_return']:.2f}%)", color='black', linestyle='--', lw=2)
plt.plot(sim_dates_adjusted, model_result['portfolio_values'], label=f"GRU Model ({model_result['total_return']:.2f}%)", color='royalblue', lw=3)

plt.title("Portfolio Value: Model vs. Buy & Hold", fontsize=18, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()